In [56]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../rna-seq/00.diff/ctl-exp.tsv", sep="\t")
df.columns = ["gene_id", "baseMean", "log2FoldChange", "lfcSE", "stat", "pvalue", "padj"]
df_AB = pd.read_csv("100kb_ABgene_id")
df_AB.columns = ["gene_id"]
df_AB 

,gene_id
0,ENSSSCG00000000361
1,ENSSSCG00000000362
2,ENSSSCG00000000363
3,ENSSSCG00000000364
4,ENSSSCG00000000453
...,...
1787,ENSSSCG00000051641
1788,ENSSSCG00000051664
1789,ENSSSCG00000051674
1790,ENSSSCG00000051774


In [57]:
df_AB1 = df.loc[df['gene_id'].isin(df_AB['gene_id'])].drop(["gene_id", "baseMean", "lfcSE", "stat", "pvalue", "padj"], axis=1)
df_AB1

,log2FoldChange
155,-0.706814
156,-0.660247
157,-0.376615
196,-0.291435
219,0.036542
...,...
19736,0.374626
19761,-0.531184
19770,-2.810966
19816,0.120955


In [58]:
df_BA = pd.read_csv("100kb_BAgene_id")
df_BA.columns = ["gene_id"]
df_BA 

,gene_id
0,ENSSSCG00000000519
1,ENSSSCG00000000555
2,ENSSSCG00000000559
3,ENSSSCG00000000631
4,ENSSSCG00000000633
...,...
819,ENSSSCG00000051454
820,ENSSSCG00000051594
821,ENSSSCG00000051680
822,ENSSSCG00000051701


In [59]:
df_BA1 = df.loc[df['gene_id'].isin(df_BA['gene_id'])].drop(["gene_id", "baseMean", "lfcSE", "stat", "pvalue", "padj"], axis=1)
df_BA1

,log2FoldChange
225,0.372533
243,-0.658243
244,2.043882
265,0.227073
266,-0.061755
...,...
19540,0.388569
19606,0.415527
19644,-0.390537
19773,-0.984589


In [85]:
df_p = pd.concat([df_AB1,df_BA1], axis=1)
df_p.columns = ["A2B", "B2A"]
df_p1 = df_p.melt()
df_p1.columns = ["Type","log2FoldChange"]
df_p1

,Type,log2FoldChange
0,A2B,-0.706814
1,A2B,-0.660247
2,A2B,-0.376615
3,A2B,-0.291435
4,A2B,0.036542
...,...,...
3363,B2A,0.388569
3364,B2A,0.415527
3365,B2A,-0.390537
3366,B2A,-0.984589


In [101]:
from scipy.stats import ranksums
z, p = ranksums(df_p['A2B'].dropna(), df_p['B2A'].dropna())
print(z, p)

-3.09813698209357 0.0019474139215634148


In [95]:
df_p['A2B'].dropna()

155     -0.706814
156     -0.660247
157     -0.376615
196     -0.291435
219      0.036542
           ...   
19736    0.374626
19761   -0.531184
19770   -2.810966
19816    0.120955
19819    0.572094
Name: A2B, Length: 1185, dtype: float64

In [96]:
df_p['B2A'].dropna()

2651     0.429677
4045     0.572094
4181    -0.281237
7095     1.958032
7114    -0.092032
           ...   
19540    0.388569
19606    0.415527
19644   -0.390537
19773   -0.984589
19807    4.057351
Name: B2A, Length: 513, dtype: float64

In [103]:
import plotly.express as px
import plotly.io as pio
import numpy as np

fig = px.box(df_p1, 
             x="Type",
             y="log2FoldChange", 
             color="Type",
             notched=True
            )

fig.update_layout(
    width=500,
    height=500,
    paper_bgcolor='white',
    plot_bgcolor='white'
)

# 添加标题，并设置标题的位置
fig.update_layout(
    title={
        'text': "A2B Vs B2A (pvalue=0.0019)",
        'y': 0.9,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    }
)

# 添加坐标轴，并设置坐标轴线为实线
fig.update_layout(
    xaxis=dict(
        showline=True,
        showgrid=True,
        zeroline=True,
        zerolinewidth=2,
        zerolinecolor='#000000'
    ),
    yaxis=dict(
        showline=True,
        showgrid=True,
        zeroline=True,
        zerolinewidth=2,
        zerolinecolor='#000000'
    )
)
fig.update_yaxes(range=[-5, 5])

# 将图像保存为PDF文件
pio.write_image(fig, 'ABboxplot.pdf')
fig.show()